# INF201 - Lecture 11
16th Nov 2026

## Topics
* Testing
    * Assertions
    * Exceptions
    * Pytest
* Advanced Python concepts
    * Factories and closures
    * Decorators and memoization
    * Iterators and generators


<img src="relatable_from_reddit.png">

## Testing: Staying in control

### Background

- Computers can solve complex tasks fast
- Humans tend to trust in results provided by computers
- In some situations, lives depend on computers working correctly
- Requires reliable software
- Difficult to achieve: we can demonstrate the presence of bugs, proving their absence is (essentially) impossible
- Field of software engineering: *Verification* and *Validation*
- We look only at essential elements

### Elements of reliable software

- Software shall not return incorrect results
- Software shall fail in controlled ways 
- Software shall handle unforseen conditions
- Software shall be tested solidly
- **Software should fail rather than return incorrect results.**

### Techniques towards reliability

- **Assertions**
    - check that requirements are fulfilled
    - stop execution if requirement not fulfilled
    - key use cases
        - very simple "emergency stops" if we don't want to spend time on proper error handling
        - catching things that "cannot happen", but where we want to be on the safe side (in large projects, you never know ...)

- **Exceptions**
    - mechanism for signaling that something unexpected happended
    - available in most modern programming languages
    - exceptions are *raised* or *thrown* when a problem is detected
    - exceptions can be *caught* and *handled*, e.g., by issuing a useful error message
    - in some languages, e.g., Python, exceptions are also used as part of normal programming

- **Testing**
    - systematic testing of code can help us to find errors
    - a proper set of tests also helps us to avoid introducing new errors as software evolves
    - *unit tests* are tests of small parts of code, typically functions
    - *integration tests* test that the parts of a larger project work together
    - *regression tests* are added when a bug is discovered
        - the test reproduces the bug
        - when the bug is fixed, the test passes
        - we keep the test, in case we should re-introduce the bug by a later change (regress)

## Assertions

- *pass* if a boolean expression is True
- *fail* if a boolean expression is False

In [5]:
assert True

In [6]:
assert False

AssertionError: 

We can use them to catch certain conditions

In [7]:
def inverse(x: int | float) -> int | float:
    """Returns 1 / x."""

    assert x != 0

    return 1 / x

In [8]:
inverse(10)  # x != 0 is True here

0.1

In [9]:
inverse(0)  # x != 0 is False here

AssertionError: 

We can provide some more information to the user by adding a string after the boolean expression:

In [10]:
def inverse(x: int | float) -> int | float:
    """Returns 1 / x."""

    assert x != 0, "Inverse of 0 is not defined."

    return 1 / x

In [11]:
inverse(0)

AssertionError: Inverse of 0 is not defined.

We can also use this to check for conditions that are mathematically defined, but make no sense.

In [12]:
import math


def area(radius: int | float) -> int | float:
    """Returns area of circle with the given radius."""

    assert radius >= 0, 'Circle radius must be positive.'

    return math.pi * radius**2

In [13]:
area(-1)

AssertionError: Circle radius must be positive.

- Assertions are the simplest, most brute-force way of checking conditions
- Typically used for errors that "cannot happen", but where we want to be safe

A note on floating-point numbers:

## Exceptions

- Exceptions provide more fine-grained control over unexpected situations
- Python defines a number of different exception types (see [Python Documentation](https://docs.python.org/3/library/exceptions.html))
- These exception types are arranged as a class hierarchy
- The diagram shows some of the pre-defined exception types

        +-- Exception
              +-- StandardError
              |    +-- ArithmeticError
              |    |    +-- FloatingPointError
              |    |    +-- OverflowError
              |    |    +-- ZeroDivisionError
              |    +-- AssertionError
              |    +-- AttributeError
              |    +-- EnvironmentError
              |    |    +-- IOError
              |    +-- EOFError
              |    +-- ImportError
              |    +-- LookupError
              |    |    +-- IndexError
              |    |    +-- KeyError
              |    +-- NameError
              |    +-- RuntimeError
              |    |    +-- NotImplementedError
              |    +-- SyntaxError
              |    |    +-- IndentationError
              |    |         +-- TabError
              |    +-- SystemError
              |    +-- TypeError
              |    +-- ValueError
              
- We can use this for our `area` function:
    - We `raise` the exception: execution stops here
    - The type of exception indicates the kind of problem
    - We can provide an error message to be sent to the user

In [14]:
def area(radius: int | float) -> int | float:
    """Returns area of circle with the given radius."""

    if radius < 0:
        raise ValueError('Circle radius must be positive.')

    return math.pi * radius**2

In [15]:
area(1)

3.141592653589793

In [16]:
area(-1)  # raises the ValueError

ValueError: Circle radius must be positive.

- Almost the same effect as an assertion

### Catching exceptions

- We can catch an exception and handle it

In [17]:
while True:
    r = float(input('Radius: '))
    if r == 0:
        break
    try:
        print('    Area:', area(r))
    except ValueError:
        print('    An error occured')  # code will keep running after printing the message

    Area: 3.141592653589793
    An error occured
    Area: 314.1592653589793
    An error occured


- We can also extract the error message an print it

In [18]:
while True:
    r = float(input('Radius: '))
    if r == 0:
        break
    try:
        print('    Area:', area(r))
    except ValueError as err:
        print(err)

    Area: 3.141592653589793
Circle radius must be positive.
    Area: 314.1592653589793
Circle radius must be positive.


- Or even nicer for the user

In [19]:
while True:
    r = float(input('Radius: '))
    if r == 0:
        break
    try:
        print('    Area:', area(r))
    except ValueError as err:
        print(f'    ERROR: {err}\n    Please try again!')

    Area: 3.141592653589793
    ERROR: Circle radius must be positive.
    Please try again!


### Separation of error detection and handling

- We *raise* an exception at the point in the code where we detect a problem.
    - Example: in the `area()` function
- We *handle* the exception where we best can regain control, e.g., by "talking" to the user
- This can be rather far away from where the error is raised.

### Exceptions as part of normal programming (Python style)

- In certain cases, we can use exceptions to choose action
- First try something, then something else
- Consider a string containing numbers

        2 3.4 12.8 72
        
- Converting this directly into numbers would force us to make all floats

In [20]:
s_list = "2 3.4 12.8 72"
[float(num) for num in s_list.split()]

[2.0, 3.4, 12.8, 72.0]

- We can try to convert to `int` first and only convert to `float` if that fails

In [21]:
def int_or_float(s: str) -> int | float:
    """Convert single-character string to int or (if that fails) float"""
    try:
        return int(s)
    except ValueError:
        return float(s)


s_list = "2 3.4 12.8 72 inf -inf nan"
[int_or_float(num) for num in s_list.split()]

[2, 3.4, 12.8, 72, inf, -inf, nan]

- We have already used this when reading files with different encodings

In [22]:
from pathlib import Path


def read_raw_log(file_path: Path | str) -> tuple[str, str]:
    """Reads file content using UTF-8, falling back to ISO-8859-1."""
    path = Path(file_path)
    try:
        return path.read_text(encoding="utf-8"), "UTF-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "ISO-8859-1"

## Testing Python code

- Systematic testing is essential part of quality control
- Do not trust code that comes without tests!
- But: passings tests are no guarantee that everything is correct
    - Tests might not cover all code
    - Tests may not cover all possible situations
    - Tests may pass for the wrong reasons
- Different levels of testing
    - Unit tests: test "units", i.e., functions and methods
    - Integration tests: test larger parts, e.g., modules or packages
    - Acceptance tests: tests by client required for accepting delivery, test the entire system against client use-cases
    - Regression tests: test detecting bugs in earlier versions, kept to avoid falling back to old mistakes

### Agile software development

- [Agile software development](https://en.wikipedia.org/wiki/Agile_software_development) is a modern (2001-) set of software development methods
- Focus on quick delivery, frequent updates, and flexibility, while maintaining quality
- Strong focus on testing
- [Test-driven development](https://en.wikipedia.org/wiki/Test-driven_development) is part of Agile
    - Write tests first, otherwise you'll never write them
    - Writing tests for working code often results in weak tests
    - Therefore
        1. Write test
        1. Write code only when a test fails
        1. Write code until tests passes (and no more)
        1. Review and refactor code without breaking tests
        1. Go back to 1
    - Important: Always run all tests after code changes
- Advantages
    - We can be much more confident in our code
    - We can make changes, big and small, and immediately check that the code still works correctly

### Tools for testing

- Systems for automatically running tests on changes or commits
    - often known as [continuous integration](https://en.wikipedia.org/wiki/Continuous_integration)
    tools
    - "watch" VCS repository and run tests on each commit or push
    - notify developers in case of trouble
    - can be combined with code-review platforms
    - Example: [GitHub Actions](https://github.com/features/actions)

### Writing tests

There are several tools (frameworks) for writing and running tests in Python

- [unittest](https://docs.python.org/3/library/unittest.html)
    - advantage: part of standard Python
    - disadvantage: no automated test discovery
- [pytest](https://docs.pytest.org/en/latest/index.html)
    - automated test discovery
    - easy coding style
    - actively developed
    - powerful advanced features
- We will use pytest 

### pytest basics

- pytest finds tests automatically
- Any file named `test_*.py` or `*_test.py` will be considered a collection of tests
- In such files
    - any function called `test_*`
    - any method called `test_*` in a class called `Test*`
  will be run as tests
- See [pytest documentation](https://docs.pytest.org/en/latest/goodpractices.html) for more on test discovery
- A test *passes* if it does not throw an exception
- In pytest, all checks are implemented as `assert`
- Module `pytest` provides useful tools

### Tests and randomness

Tests including random numbers introduce particular problems; we will not discuss these here.

### A Pytest example

#### Preparations

- You first need to `pip install pytest pytest-coverage`

#### Running a first test

- Test examples are in directory `tests`
- We will start with `test_example_buggy.py` 
- Run this set of tests in the terminal with
    ```
    cd lectures/lecture_11
    pytest tests/test_example_buggy.py
    ```
- On my computer, I got the following results

``` 
================================= test session starts =================================
platform win32 -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: C:\Users\tutor5923\Documents\inf201-h26-my-repo
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.12.2
collected 2 items                                                                      

tests\test_example_buggy.py .F                                                   [100%]

====================================== FAILURES =======================================
____________________________________ test_square_2 ____________________________________

    def test_square_2() -> None:
>       assert square(2) == 4
E       assert 8 == 4
E        +  where 8 = square(2)

tests\test_example_buggy.py:22: AssertionError
=============================== short test summary info ===============================
FAILED tests\test_example_buggy.py::test_square_2 - assert 8 == 4
============================= 1 failed, 1 passed in 0.11s =============================
```

We can also run this in Jupyter notebook by using `!` which runs the rest of the line as a terminal command

In [24]:
!python -m pytest tests/test_example_buggy.py

============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\tutor5923\Documents\inf201-h26-my-repo
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.12.2
collected 2 items

tests\test_example_buggy.py .F                                           [100%]

================================== FAILURES ===================================
________________________________ test_square_2 ________________________________

    def test_square_2() -> None:
>       assert square(2) == 4
E       assert 8 == 4
E        +  where 8 = square(2)

tests\test_example_buggy.py:22: AssertionError
=========================== short test summary info ===========================
FAILED tests\test_example_buggy.py::test_square_2 - assert 8 == 4
========================= 1 failed, 1 passed in 0.12s =========================


- We see that one test failed, one passed
- Corrected code is in `test_example_fixed.py`
- Both tests pass for this code

In [25]:
!python -m pytest tests/test_example_fixed.py

============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\tutor5923\Documents\inf201-h26-my-repo
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.12.2
collected 2 items

tests\test_example_fixed.py ..                                           [100%]

============================== 2 passed in 0.06s ==============================


### Testing for exceptions

- In some cases, we want to test that a function raises an exception
- In pytest, with can do this with the help of a *context*
- `test_division.py` gives an example of how to test that a `ValueError` is raised for an unacceptable argument value
    ```python
        def test_inv_raises():
            with pytest.raises(ValueError):
                inv(0)
    ```
- Passes if `ValueError` (or a subclass of it) is raised inside the block.
- Fails if no exception is raised, or if a different exception type (like TypeError) is raised instead.

In [26]:
!python -m pytest tests/test_division.py

============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\tutor5923\Documents\inf201-h26-my-repo
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.12.2
collected 1 item

tests\test_division.py .                                                 [100%]

============================== 1 passed in 0.03s ==============================


### Parameterizing tests

- Often, we want to perform the same test for different argument values
- Test parameterization makes this easy in PyTest
- See also https://docs.pytest.org/en/stable/example/parametrize.html

#### Example: Testing the `square()` function

##### Explicit tests for each argument

```python
def test_square_1():
    assert square(1) == 1

def test_square_2():
    assert square(2) == 4
```

##### Parameterized tests

```python
@pytest.mark.parametrize("x, expected",
                         [[1, 1],
                          [2, 4]
                          ])
def test_square(x, expected):
    assert square(x) == expected
```

- `@pytest.mark.parametrize()` is a *decorator*
- The first decorator argument is a string with comma-separated variable names
- The second decorator argument is a list
    - Each list element represents one test case
    - Each list element must be a list (or tuple) with as many elements as variable names in the string
    - The variables need to be passed to the test function
- Pytest runs each case as a separate test
- See `test_parametrize.py` for an example
- Question: Which other cases could we add to the tests for `square()`?

In [ ]:
!python -m pytest tests/test_parametrize_buggy.py

============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\tutor5923\Documents\inf201-h26-my-repo
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.12.2
collected 2 items

tests\test_parametrize.py .F                                             [100%]

================================== FAILURES ===================================
______________________________ test_square[2-4] _______________________________

x = 2, expected = 4

    @pytest.mark.parametrize("x, expected",
                             [[1, 1],
                              [2, 4]
                              ])
    def test_square(x, expected):
>       assert square(x) == expected
E       assert 8 == 4
E        +  where 8 = square(2)

tests\test_parametrize.py:24: AssertionError
=========================== short test summary info ===========================
FAILED tests\test_parametrize.py::test_square[2-

In [28]:
!python -m pytest tests/test_parametrize_fixed.py

============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.2.0, pluggy-1.7.0
rootdir: c:\Users\tutor5923\Documents\inf201-h26-my-repo
configfile: pyproject.toml
plugins: anyio-4.15.1, platformdirs-4.12.2
collected 2 items

tests\test_parametrize_fixed.py ..                                       [100%]

============================== 2 passed in 0.04s ==============================


#### A note on floating-point numbers
* Use `pytest.approx()` to avoid tests failing do to floating-point rounding errors

In [29]:
import pytest

assert 0.1 + 0.2 == pytest.approx(0.3)

In [ ]:
assert 0.1 + 0.2 == 0.3

AssertionError: 

### Test discovery

- So far, we have run individual tests from the terminal (directly or in Jupter code cells)
- Usually, we want to run a whole set of tests for a project
- PyTest provides automatic *test discovery*
    - If we run it with a directory, it will find all files starting with `test*`
    - Inside those files, it will look for all functions starting with `test*` and run them as tests
    - We can also create classes called `Test*`, but we won't look into that here
    - If we run pytest without a directory name, it will search the current directory

### Test coverage

- It is important to know how much code is actually tested
- This is called *test coverage*
- Complete coverage can be difficult to achieve in complex programs
- Even complete coverage does not guarantee absence of errors
- To measure coverage, run `pytest --cov tests`
- Here the result for the example tests:
    ``` 
    Name                          Stmts   Miss  Cover
    -------------------------------------------------
    tests/test_division.py            8      1    88%
    tests/test_example_buggy.py       6      0   100%
    tests/test_example_fixed.py       6      0   100%
    tests/test_parametrize.py         6      0   100%
    -------------------------------------------------
    TOTAL                            26      1    96%
    ```
- Note: Coverage measurement only works for directories, not for individual test files

### Pytest in Visual Studio Code

- We can run tests directly from Pytest
- Requires configuration (demo in class)
- See https://code.visualstudio.com/docs/python/testing


### Recommentation: Write code that is easy to test!

Note that several of the recommendations you have had earlier in the course for how to code well, also makes the code easier to test:
* Smaller single-responsibility functions/classes/methods
    * When we can easier define what the function is supposed to do, we can easier write the tests
* Type hints
    * Helps you think through which data types your functions should be valid for, which allows you to test them for these (and potentially raise errors for other types)
* Methods/functions return strings rather than print
    * Allows us to test that the correct string is produced
* Keeping you work well organised in well-named files and folders
    * Automatic test discovery

### Recommendation: What should I test?
* Typical cases: Standard inputs where you know the expected answer
* Boundary/edge cases: If your function input is a list, how does the function handle empty lists?
* Invalid input: What happens if somebody tried to use your `Dice` class to create a 0-sided dice?

### More advanced: Automated testing
* [GitHub Actions](https://docs.github.com/en/actions/tutorials/build-and-test-code/python) allows you to automatically test code pushed to the repository
* Defined in a yml file under `.github/workflows/` in the repository root directory 
    * See `tests.yml` for an example file
* You can view the results of the tests under the Actions tab on GitHub
    * Note: If your tests fail during "Install dependencies" this could be due to Windows specific packages in your `requirements.txt`. Fix this by removing these packages from the requirements file.
* Can also be set up to test for several operating systems and Python versions (very useful of you are making packages to be used by many others)
    * Example from GitHub's Actions documentation:
        ```
        name: Python package

        on: [push]

        jobs:
        build:

            runs-on: ${{ matrix.os }}
            strategy:
            matrix:
                os: [ubuntu-latest, macos-latest, windows-latest]
                python-version: ["3.9", "3.11", "3.13", "pypy3.10"]
                exclude:
                - os: macos-latest
                    python-version: "3.11"
                - os: windows-latest
                    python-version: "3.11"

        ```
    * See also [tox](https://tox.wiki/en/latest/index.html) for testing across OS and versions

## A selection of more advanced Python concepts

## Factories and closures

### Factory

- A function producing other functions (or classes, or objects)

In [31]:
# See multiplier_complete.py for version with doc-string and type hints
def make_multiplier(factor):

    def multiplier(n):
        return factor * n

    return multiplier


doubler = make_multiplier(2)
tripler = make_multiplier(3)

factor = 10

print(doubler(2), doubler(5), doubler('a'))
print(tripler(2), tripler(5), tripler('a'))

4 10 aa
6 15 aaa


- Function `multiplier(n)` is defined *inside* `make_multiplier`
- The variable `factor` inside `multiplier(n)` is bound *when the function definition is executed*
- Therefore, we get `multiplier` functions with different values of `factor`
- Shows up in function representation as `<locals>` dictionary
    - local variables in factory class at time when `multiplier` was defined

In [32]:
doubler

<function __main__.make_multiplier.<locals>.multiplier(n)>

### Closure
- Here, `__main__.multiplier_factory.<locals>.multiplier` forms a *closure*
- `<locals>` stores variables bound in the environment in which `multiplier` was defined
- Generally: A closure binds free variables in a function to the value to which the name was bound when the function was defined, i.e., when the closure was created
- See also <a href="https://en.wikipedia.org/wiki/Closure_(computer_programming)">Wikipedia article on Closure</a>

#### Example

##### Function with free variable without closure

- Note: in the definition below, `foo` is not bound, it is a *free* variable

In [33]:
def f_open(x):
    return foo * x

- We cannot use `f_open()` unless `foo` is defined somewhere
- When executed, it will use the most local `foo` value available
- Can lead to difficult to predict results

In [34]:
f_open(10)

NameError: name 'foo' is not defined

In [35]:
foo = 5
f_open(10)

50

In [36]:
def g(y):
    foo = 100
    return f_open(y)

foo = 7
g(3)

21

How Python resolves `foo` (the LEGB rule):
* When `f_open(3)` executes, Python searches for `foo` in four scopes in strict LEGB order (Local $\rightarrow$ Enclosing $\rightarrow$ Global $\rightarrow$ Built-in):
    * Local (`f_open`): Is `foo` assigned inside `f_open`? No.
    * Enclosing: Was `f_open` defined inside another function? No, it was written at the module's top level. Even though `g` called `f_open`, `g`'s call frame is not an enclosing scope (`f_open` was not *defined* inside of `g`).
    * Global: Is `foo` defined at the module level? Yes (`foo` = 7). Python stops searching and uses 7.
* Inside `g`, the line `foo = 100` creates a completely separate local variable that is only visible to code written inside `g`'s indented block.
* Python (and many other programming languages) use <a href="https://en.wikipedia.org/wiki/Scope_(computer_science)#Lexical_scoping">*lexical scoping*</a>: name lookup is determined by where a function is *defined* in the program text, not where it is used

        
##### Function with free variable with closure

- In definition below, `foo` is free in `f_func` itself
- By defining `f_func` inside another function, here `f_factory`, `foo` in `f_func` is bound to the `foo` argument of `f_factory` forming a closure

In [ ]:
def f_factory(foo):
    def f_func(x):
        return foo * x
    return f_func


f3 = f_factory(3)
f8 = f_factory(8)

f3(9), f8(9)

(27, 72)

[Code visualised with Pythontutor](https://pythontutor.com/visualize.html#code=def%20f_factory%28foo%29%3A%0A%20%20%20%20def%20f_func%28x%29%3A%0A%20%20%20%20%20%20%20%20return%20foo%20*%20x%0A%20%20%20%20return%20f_func%0A%0Af3%20%3D%20f_factory%283%29%0Af8%20%3D%20f_factory%288%29%0Aprint%28f3%289%29,%20f8%289%29%29%0A&cumulative=false&curInstr=18&heapPrimitives=nevernest&mode=display&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false)